In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Menetapkan Daftar Kolom Ekspor di Data Dictionary**

In [17]:
import os

# 1. Pastikan folder 'docs' sudah tersedia di direktori proyekmu
os.makedirs("docs", exist_ok=True)

# 2. Membuat file data_dictionary_v1.md secara otomatis
with open("docs/data_dictionary_v1.md", "w", encoding="utf-8") as f:
    f.write("""# Kamus Data (Data Dictionary v1.0)
## Bagian Kolom Ekspor (Export Schema Contract)

Berikut adalah daftar 21 kolom resmi yang menjadi kontrak baku untuk fitur ekspor data (`EXPORT_COLUMNS`):

| No | Nama Kolom (`EXPORT_COLUMNS`) | Deskripsi / Keterangan |
| :---: | :--- | :--- |
| 1 | `comment_id` | ID unik komentar |
| 2 | `post_id` | ID unik unggahan/postingan terkait |
| 3 | `platform` | Platform media sosial (Instagram, YouTube, dll.) |
| 4 | `source_account` | Akun sumber atau channel pemilik unggahan |
| 5 | `post_url` | Tautan/URL menuju unggahan asli |
| 6 | `published_at` | Tanggal dan waktu unggahan dipublikasikan |
| 7 | `context_text` | Teks konteks atau deskripsi unggahan |
| 8 | `content_format` | Format konten (video, carousel, teks, dll.) |
| 9 | `content_category` | Kategori isi konten (Infrastruktur, Layanan Publik, dll.) |
| 10 | `commented_at` | Tanggal dan waktu komentar ditulis |
| 11 | `text_original` | Teks asli komentar dari pengguna |
| 12 | `split` | Pembagian data (train/test/validation) |
| 13 | `data_version` | Versi dataset yang digunakan |
| 14 | `pred_label` | Label hasil prediksi model terbaru |
| 15 | `pred_score` | Skor keyakinan (*confidence score*) prediksi model |
| 16 | `score_type` | Jenis skor (probability, logit, dll.) |
| 17 | `model_version` | Versi model *machine learning* yang mengeksekusi prediksi |
| 18 | `verified_label` | Label hasil validasi/verifikasi manual terbaru |
| 19 | `verify_reason` | Alasan atau catatan koreksi dari validator |
| 20 | `reviewer_id` | ID unik validator/reviewer yang memeriksa |
| 21 | `verified_at` | Waktu saat verifikasi manual dilakukan |

*Kebijakan Keamanan:* Kolom identitas individu seperti `username`, `user_name`, atau `author` **dilarang keras** disertakan dalam ekspor data demi menjaga privasi.
""")

print("File docs/data_dictionary_v1.md berhasil dibuat dan diperbarui!")

File docs/data_dictionary_v1.md berhasil dibuat dan diperbarui!


**Membuat Fungsi export_filtered(conn, filters) di src/export.py**

In [18]:
import os

# Memastikan folder 'src' ada
os.makedirs("src", exist_ok=True)

content_export = '''import pandas as pd
from src.queries import filter_comments

# Kontrak daftar kolom ekspor yang wajib dipatuhi (21 kolom)
EXPORT_COLUMNS = [
    "comment_id", "post_id", "platform", "source_account", "post_url", "published_at",
    "context_text", "content_format", "content_category", "commented_at", "text_original",
    "split", "data_version", "pred_label", "pred_score", "score_type", "model_version",
    "verified_label", "verify_reason", "reviewer_id", "verified_at"
]

def export_filtered(conn, filters=None):
    # Mengambil data yang telah difilter dan merapikan kolom sesuai kontrak EXPORT_COLUMNS tanpa PII.
    try:
        df_filtered = filter_comments(conn)

        # Terapkan filter tambahan secara aman menggunakan Pandas jika parameter filters tersedia
        if filters and not df_filtered.empty:
            if "platform" in filters and filters["platform"] and filters["platform"] != "Semua":
                if "platform" in df_filtered.columns:
                    df_filtered = df_filtered[df_filtered["platform"] == filters["platform"]]
            if "content_category" in filters and filters["content_category"] and filters["content_category"] != "Semua":
                if "content_category" in df_filtered.columns:
                    df_filtered = df_filtered[df_filtered["content_category"] == filters["content_category"]]
    except Exception as e:
        # Fallback jika terjadi kendala pada fungsi queries
        query = "SELECT * FROM comments"
        df_filtered = pd.read_sql(query, conn)

    # Jika dataframe kosong, kembalikan kerangka kolom kontrak
    if df_filtered.empty:
        return pd.DataFrame(columns=EXPORT_COLUMNS)

    # Pastikan tidak ada kolom identitas pengguna (PII) yang ikut terekspor
    pii_cols = ["username", "user_name", "author", "user_id"]
    df_filtered = df_filtered.drop(columns=[c for c in pii_cols if c in df_filtered.columns])

    # Reindex kolom agar urutan dan nama selalu konsisten (kolom kosong otomatis bernilai NaN/kosong)
    out = df_filtered.reindex(columns=EXPORT_COLUMNS)

    # Validasi pengaman (Assertion) untuk memastikan keamanan privasi
    assert not any(c in out.columns for c in ["username", "user_name", "author"]), "Pelanggaran privasi: Kolom PII terdeteksi!"

    return out
'''

# Tulis ulang file src/export.py dengan format bersih
with open("src/export.py", "w", encoding="utf-8") as f:
    f.write(content_export)

print("File src/export.py berhasil diperbarui dan bebas dari error indentasi!")

File src/export.py berhasil diperbarui dan bebas dari error indentasi!


**Membuat app/pages/6_Ekspor.py di Colab**

In [19]:
import os

# 1. Pastikan folder 'app/pages' sudah tersedia
os.makedirs("app/pages", exist_ok=True)

# 2. Susun kode lengkap untuk halaman Streamlit Ekspor Data dengan perbaikan sys.path
content_ekspor = """import streamlit as st
import sqlite3
import pandas as pd
import sys
import os

# Menambahkan direktori root proyek ke sys.path agar folder 'src' terbaca
ROOT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../"))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.export import export_filtered

st.set_page_config(page_title="Ekspor Data", page_icon="📥", layout="wide")

st.title("📥 Ekspor Data Analisis & Verifikasi")
st.markdown("Unduh dataset hasil saringan analisis sentimen dan validasi manual sesuai parameter filter aktif.")

# Koneksi database
@st.cache_resource
def get_connection():
    return sqlite3.connect("data/sentiment.db", check_same_thread=False)

conn = get_connection()

# Sidebar Filter untuk Ekspor
st.sidebar.header("Filter Ekspor")
platform_opt = st.sidebar.selectbox("Pilih Platform", ["Semua", "Instagram", "YouTube", "TikTok", "Twitter"])
category_opt = st.sidebar.selectbox("Pilih Kategori", ["Semua", "Infrastruktur", "Layanan Publik", "Ekonomi", "Pendidikan"])

filters = {}
if platform_opt != "Semua":
    filters["platform"] = platform_opt
if category_opt != "Semua":
    filters["content_category"] = category_opt

# Ambil data menggunakan fungsi export_filtered
df_export = export_filtered(conn, filters=filters)

# Metrik ringkasan baris
col1, col2 = st.columns(2)
with col1:
    st.metric("Total Baris Data Siap Ekspor", f"{len(df_export):,}")
with col2:
    st.metric("Jumlah Kolom Kontrak", f"{len(df_export.columns)}")

if not df_export.empty:
    st.subheader("👀 Pratinjau 5 Baris Pertama")
    st.dataframe(df_export.head(5), use_container_width=True)

    # Konversi ke CSV untuk tombol unduh
    csv_bytes = df_export.to_csv(index=False).encode('utf-8')

    st.download_button(
        label="📥 Unduh Berkas CSV (Filtered Export)",
        data=csv_bytes,
        file_name="dataset_sentiment_filtered.csv",
        mime="text/csv",
        help="Klik untuk mengunduh data dalam format CSV standar sesuai kontrak."
    )
else:
    st.warning("⚠️ Tidak ada data yang ditemukan berdasarkan kombinasi filter tersebut.")
"""

# 3. Tulis ulang file app/pages/6_Ekspor.py
with open("app/pages/6_Ekspor.py", "w", encoding="utf-8") as f:
    f.write(content_ekspor)

print("File app/pages/6_Ekspor.py berhasil diperbarui dengan penanganan path sys!")

File app/pages/6_Ekspor.py berhasil diperbarui dengan penanganan path sys!


**Membuat File Laporan Pemeriksaan Ekspor**

In [20]:
import os

# 1. Pastikan folder 'reports/evidence' sudah tersedia
os.makedirs("reports/evidence", exist_ok=True)

# 2. Isi teks untuk laporan pemeriksaan D26
content_laporan = """# Laporan Pemeriksaan Ekspor Data (D26)
*Tanggal: 11 November 2026*
*Nama: Azzahra Attaqina*

## 1. Ringkasan Pemeriksaan Kontrak Ekspor
* **Jumlah Kolom:** 21 kolom sesuai dengan kontrak baku `EXPORT_COLUMNS`.
* **Privasi PII:** Terverifikasi aman dan bebas dari atribut identitas pengguna (`username`, `user_name`, `author`).
* **Konsistensi Baris:** Jumlah baris dan pratinjau data di halaman `6_Ekspor.py` berjalan akurat sesuai parameter filter aktif.

## 2. Hasil Pemeriksaan Sampel dan Kesesuaian Database
* **Validasi Kontrak Kolom:** Seluruh 21 kolom terindeks dengan benar; kolom yang tidak memiliki data tetap dipertahankan dalam bentuk kosong (*NaN/Blank*) untuk menjaga struktur kontrak.
* **Kesesuaian 3 Baris Acak:** Berdasarkan pengujian sampel data acak dari fungsi `export_filtered`, label prediksi (`pred_label`) dan struktur metrik 100% konsisten dengan data asli di basis data SQLite (`data/sentiment.db`).
"""

# 3. Tulis ke file reports/evidence/D26_ekspor_cek.md
with open("reports/evidence/D26_ekspor_cek.md", "w", encoding="utf-8") as f:
    f.write(content_laporan)

print("File reports/evidence/D26_ekspor_cek.md berhasil dibuat!")

File reports/evidence/D26_ekspor_cek.md berhasil dibuat!
